# Vis-Head Routing + Causal Steering Across Qwen3-VL Training Stages

The Qwen2-VL comparison (`top2_phrasings_across_model_stages.ipynb`) used a
genuine base/instruct/agentic triad, because all three exist as official
checkpoints at that size. For **Qwen3-VL**, no official base (pretrained-only,
non-instruction-tuned) checkpoint is public, and no official agentic/GUI-tuned
checkpoint exists either — only unverified community fine-tunes, which are not
used here (see project history for why: unverified authorship/quality, no
confirmed matching architecture).

What Qwen **does** officially release at matching size (8B) is two distinct
post-training stages built on the same base:

- **instruct** — `Qwen/Qwen3-VL-8B-Instruct` (standard instruction-following tune)
- **thinking** — `Qwen/Qwen3-VL-8B-Thinking` (reasoning/chain-of-thought tune)

Both confirmed identical architecture (36 layers x 32 heads, `model_type=qwen3_vl`)
via `AutoConfig`, so head indices are directly comparable across the two —
unlike a cross-family comparison, there's no risk of an index referring to a
structurally different head.

Same design as the Qwen2-VL notebook: `find`/`identify` phrasings (the top-2 causal
phrasings from the 10-phrasing sweep), no-cue baseline, word-position location-cue,
and attention-steered MCQ accuracy, all reused unchanged from the validated
pipeline.

In [1]:
%matplotlib inline
import re
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis_head").exists():
    raise RuntimeError("Run this notebook from the repository root (vis-head/).")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

from vis_head.common import DEFAULT_SEED, dump_json, make_output_paths
from vis_head.gaze import aggregate_region_attention, collect_last_query_attentions, rank_heads_by_score
from vis_head.imagenet_grid import DEFAULT_IMAGENET_ROOT, mcq_prompt, PROMPT_TEMPLATES, list_val_class_dirs, load_class_names, sample_grid
from vis_head.modeling import decode_generated_text, find_image_token_range, load_model_and_processor, model_dims, prepare_inputs, run_generation
from vis_head.regions import assign_grid_cells_to_tokens, region_positions_from_ids
from vis_head.steering import group_heads_by_layer, intervention_positions, make_static_attention_mask_hook, register_mask_hooks, remove_handles

MODELS = {
    "instruct": "Qwen/Qwen3-VL-8B-Instruct",
    "thinking": "Qwen/Qwen3-VL-8B-Thinking",
}
PHRASINGS = {
    "find": lambda name: PROMPT_TEMPLATES["find"].format(name=name),
    "identify": lambda name: PROMPT_TEMPLATES["identify"].format(name=name),
}
DEVICE = "cuda:0"
SEED = DEFAULT_SEED

ROWS, COLS = 2, 2
N_CELLS = ROWS * COLS
CELL_SIZE = 256
N_DISCOVERY_SAMPLES = 150
TOP_K_HEADS = 15
N_MCQ_SAMPLES = 150
N_OPTIONS = 4
CAUSAL_MAX_NEW_TOKENS_BY_MODEL = {"instruct": 6, "thinking": 600}   # Thinking emits an extended reasoning trace before the answer letter
OPTION_LETTERS = ["A", "B", "C", "D"][:N_OPTIONS]

imagenet_class_dirs = list_val_class_dirs(DEFAULT_IMAGENET_ROOT)
imagenet_class_names = load_class_names(DEFAULT_IMAGENET_ROOT)

print(f"Models: {list(MODELS.keys())}")
print(f"Phrasings: {list(PHRASINGS.keys())}")

# Both checkpoints are Instruct-family (Thinking is also chat-template-capable),
# so unlike the Qwen2-VL base-model case, no chat-template workaround is needed --
# each model's own processor.apply_chat_template works directly.

Models: ['instruct', 'thinking']
Phrasings: ['find', 'identify']


## Discovery + causal helper functions (identical to `top2_phrasings_across_model_stages.ipynb`)

In [2]:
def discover_vis_head_scores(model, processor, n_layers, n_heads, spatial_merge, prompt_fn, label, seed):
    rng = np.random.RandomState(seed)
    raw_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    valid = 0
    for _ in tqdm(range(N_DISCOVERY_SAMPLES), desc=f"[{label}]", leave=False):
        grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng,
                            class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
        target_cell = int(rng.randint(N_CELLS))
        prompt = prompt_fn(grid.cell_names[target_cell])
        try:
            inputs = prepare_inputs(processor, grid.grid, prompt, DEVICE)
            region_ids, _ = assign_grid_cells_to_tokens(
                image_grid_thw=inputs["image_grid_thw"], rows=ROWS, cols=COLS, spatial_merge=spatial_merge)
            attn_at_query = collect_last_query_attentions(model, inputs)
            region_attention = aggregate_region_attention(
                attn_at_query=attn_at_query, inputs=inputs, processor=processor, region_ids=region_ids, n_regions=N_CELLS)
            raw = region_attention[:, :, target_cell]
            raw_sum += raw
            valid += 1
        except Exception as exc:
            print(f"Skipping grid: {exc}")
    print(f"  [{label:>10s}] valid={valid}/{N_DISCOVERY_SAMPLES}  mean score={raw_sum.mean() / max(valid, 1):.5f}")
    return (raw_sum / max(valid, 1)).astype(np.float32)


def build_mcq_sample(rng):
    grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng,
                        class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
    target_cell = int(rng.randint(N_CELLS))
    correct_name = grid.cell_names[target_cell]
    other_names = [n for i, n in enumerate(grid.cell_names) if i != target_cell]
    n_distractors = min(N_OPTIONS - 1, len(other_names))
    distractor_idx = rng.choice(len(other_names), size=n_distractors, replace=False)
    distractors = [other_names[i] for i in distractor_idx]
    options = [correct_name] + distractors
    order = rng.permutation(len(options))
    options = [options[i] for i in order]
    correct_letter = OPTION_LETTERS[int(np.where(order == 0)[0][0])]
    option_lines = "  ".join(f"{letter}) {name}" for letter, name in zip(OPTION_LETTERS[:len(options)], options))
    prompt = f"Which of the following is shown in this image? {option_lines}. Answer with only the letter."
    location_prompt = mcq_prompt(target_cell + 1, ROWS, COLS, options, OPTION_LETTERS[:len(options)])
    return {"grid": grid, "target_cell": target_cell, "options": options, "correct_letter": correct_letter,
            "prompt": prompt, "location_prompt": location_prompt}


def extract_letter(text, valid_letters):
    """Return the LAST standalone occurrence of a valid letter, not the first --
    a chain-of-thought response (e.g. Qwen3-VL-Thinking) mentions multiple option
    letters while reasoning before concluding; the first match is often wrong."""
    matches = re.findall(r"\b([" + "".join(valid_letters) + r"])\b", text.upper())
    return matches[-1] if matches else None


def run_mcq(model, processor, n_heads, spatial_merge, sample, heads_by_layer, prompt_key="prompt", max_new_tokens=6):
    inputs = prepare_inputs(processor, sample["grid"].grid, sample[prompt_key], DEVICE)
    prompt_length = int(inputs["input_ids"].shape[1])
    handles = []
    if heads_by_layer is not None:
        img_start, img_end = find_image_token_range(inputs, processor)
        region_ids, _ = assign_grid_cells_to_tokens(
            image_grid_thw=inputs["image_grid_thw"], rows=ROWS, cols=COLS, spatial_merge=spatial_merge)
        positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
        target_positions = positions[sample["target_cell"]]
        other_positions = [p for i in range(N_CELLS) if i != sample["target_cell"] for p in positions[i]]
        suppress_positions, boost_positions, pad = intervention_positions(
            mode="boost_suppress", target_positions=target_positions, other_image_positions=other_positions,
            img_start=img_start, img_end=img_end, prompt_length=prompt_length)
        hook_by_layer = {
            layer_idx: make_static_attention_mask_hook(
                head_indices=heads, suppress_positions=suppress_positions, boost_positions=boost_positions,
                n_query_heads=n_heads, device=DEVICE, decode_only=False, pad_with_suppress=pad)
            for layer_idx, heads in heads_by_layer.items()
        }
        handles = register_mask_hooks(model, hook_by_layer)
    try:
        sequences = run_generation(model=model, inputs=inputs, max_new_tokens=max_new_tokens)
    finally:
        remove_handles(handles)
    text = decode_generated_text(processor, sequences, prompt_length)
    valid_letters = OPTION_LETTERS[: len(sample["options"])]
    predicted = extract_letter(text, valid_letters)
    return {"text": text, "predicted": predicted, "correct": predicted == sample["correct_letter"]}

## Run: discovery + causal eval, per model, per phrasing

In [3]:
results = {}   # (model_tag, phrasing) -> dict
for model_tag, model_id in MODELS.items():
    print(f"\n=== Loading {model_id} ({model_tag}) ===")
    model, processor = load_model_and_processor(model_id=model_id, device=DEVICE)
    n_layers, n_heads, spatial_merge = model_dims(model)
    print(f"{n_layers} layers x {n_heads} heads")

    vis_head_scores = {}
    vis_head_ranked = {}
    for tag, fn in PHRASINGS.items():
        scores = discover_vis_head_scores(model, processor, n_layers, n_heads, spatial_merge, fn, f"{model_tag}/{tag}", seed=SEED + 100)
        vis_head_scores[tag] = scores
        vis_head_ranked[tag] = rank_heads_by_score(scores)

    max_new_tokens = CAUSAL_MAX_NEW_TOKENS_BY_MODEL[model_tag]
    mcq_rng = np.random.RandomState(SEED + 555)
    mcq_samples = [build_mcq_sample(mcq_rng) for _ in range(N_MCQ_SAMPLES)]

    shared_baseline_res, shared_location_res = [], []
    for sample in tqdm(mcq_samples, desc=f"MCQ [{model_tag}] baseline+location", leave=False):
        shared_baseline_res.append(run_mcq(model, processor, n_heads, spatial_merge, sample, None, "prompt", max_new_tokens))
        shared_location_res.append(run_mcq(model, processor, n_heads, spatial_merge, sample, None, "location_prompt", max_new_tokens))
    b_acc = np.mean([r["correct"] for r in shared_baseline_res])
    l_acc = np.mean([r["correct"] for r in shared_location_res])
    print(f"  [{model_tag}] baseline={b_acc:.3f}  location-cue={l_acc:.3f}")

    for tag, fn in PHRASINGS.items():
        heads_by_layer = group_heads_by_layer([(r["layer"], r["head"]) for r in vis_head_ranked[tag][:TOP_K_HEADS]])
        steered_res = []
        for sample in tqdm(mcq_samples, desc=f"MCQ [{model_tag}/{tag}] steered", leave=False):
            steered_res.append(run_mcq(model, processor, n_heads, spatial_merge, sample, heads_by_layer, "prompt", max_new_tokens))
        s_acc = np.mean([r["correct"] for r in steered_res])
        print(f"  [{model_tag}/{tag}] steered={s_acc:.3f}")
        results[(model_tag, tag)] = {
            "vis_head_scores": vis_head_scores[tag], "baseline": shared_baseline_res,
            "location_cue": shared_location_res, "steered": steered_res,
        }

    del model, processor
    import gc, torch
    gc.collect()
    torch.cuda.empty_cache()


=== Loading Qwen/Qwen3-VL-8B-Instruct (instruct) ===


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

36 layers x 32 heads


...

  [thinking] baseline=0.280  location-cue=0.973


MCQ [thinking/find] steered:   0%|          | 0/150 [00:00<?, ?it/s]

  [thinking/find] steered=0.240


MCQ [thinking/identify] steered:   0%|          | 0/150 [00:00<?, ?it/s]

  [thinking/identify] steered=0.260


## Results table

In [4]:
def macro_f1(results_list, samples_list):
    y_true = [s["correct_letter"] for s in samples_list]
    y_pred = [r["predicted"] if r["predicted"] is not None else "UNPARSED" for r in results_list]
    labels = sorted(set(y_true) | set(y_pred))
    return float(f1_score(y_true, y_pred, labels=labels, average="macro"))


def mcnemar_p(cond_a, cond_b):
    b = sum(1 for a, c in zip(cond_a, cond_b) if not a["correct"] and c["correct"])
    c = sum(1 for a, c in zip(cond_a, cond_b) if a["correct"] and not c["correct"])
    n_discordant = b + c
    if n_discordant == 0:
        return float("nan")
    stat = (abs(b - c) - 1) ** 2 / n_discordant
    return float(stats.chi2.sf(stat, df=1))


outputs = make_output_paths("qwen3vl_stages")
rows_summary = []
print(f"{'model':>10s}  {'phrasing':>10s}  {'vh mean':>9s}  {'vh median':>10s}  {'vh std':>9s}  "
      f"{'baseline':>9s}  {'loc-cue':>9s}  {'steered':>9s}  {'f1 base':>9s}  {'f1 loc':>9s}  {'f1 steer':>9s}  "
      f"{'p(s,b)':>10s}  {'p(s,l)':>10s}")
for model_tag in MODELS:
    for tag in PHRASINGS:
        r = results[(model_tag, tag)]
        scores_flat = r["vis_head_scores"].reshape(-1)
        b_acc = np.mean([x["correct"] for x in r["baseline"]])
        l_acc = np.mean([x["correct"] for x in r["location_cue"]])
        s_acc = np.mean([x["correct"] for x in r["steered"]])
        f1_base = macro_f1(r["baseline"], mcq_samples)
        f1_loc = macro_f1(r["location_cue"], mcq_samples)
        f1_steer = macro_f1(r["steered"], mcq_samples)
        p_sb = mcnemar_p(r["baseline"], r["steered"])
        p_sl = mcnemar_p(r["location_cue"], r["steered"])
        print(f"{model_tag:>10s}  {tag:>10s}  {scores_flat.mean():9.5f}  {np.median(scores_flat):10.5f}  "
              f"{scores_flat.std():9.5f}  {b_acc:9.3f}  {l_acc:9.3f}  {s_acc:9.3f}  "
              f"{f1_base:9.3f}  {f1_loc:9.3f}  {f1_steer:9.3f}  {p_sb:10.3e}  {p_sl:10.3e}")
        rows_summary.append({"model": model_tag, "phrasing": tag, "vh_mean": float(scores_flat.mean()),
                              "vh_median": float(np.median(scores_flat)), "vh_std": float(scores_flat.std()),
                              "baseline_acc": float(b_acc), "location_cue_acc": float(l_acc),
                              "steered_acc": float(s_acc), "f1_baseline": f1_base, "f1_location_cue": f1_loc,
                              "f1_steered": f1_steer, "p_steer_vs_base": p_sb, "p_steer_vs_loc": p_sl})

dump_json(outputs.logs_dir / "qwen3vl_stages_summary.json", rows_summary)

     model    phrasing    vh mean   vh median     vh std   baseline    loc-cue    steered    f1 base     f1 loc   f1 steer      p(s,b)      p(s,l)
  instruct        find    0.02630     0.01352    0.03609      0.280      0.993      0.667      0.279      0.993      0.661   1.041e-12   7.025e-12
  instruct    identify    0.02561     0.01263    0.03452      0.280      0.993      0.653      0.279      0.993      0.651   5.128e-13   2.534e-12
  thinking        find    0.02287     0.01314    0.02855      0.280      0.973      0.240      0.261      0.973      0.239   4.510e-01   2.675e-25
  thinking    identify    0.02211     0.01248    0.02800      0.280      0.973      0.260      0.261      0.973      0.264   7.705e-01   1.216e-24


PosixPath('/mnt/abka03/Projects/vis-head/logs/qwen3vl_stages/qwen3vl_stages_summary.json')

## Result

(filled in after running)